# 06 - Ethical AI, Privacy & Adversarial Robustness
## AI-Based Cyber Risk Detection System (Objective 11)

This notebook evaluates critical ethical and security aspects of automated cyber risk detection:
1. **Algorithmic Fairness & Bias Auditing**: Disparate Impact (Four-Fifths Rule), Demographic Parity, Equalized Odds across user locations/roles.
2. **Privacy Preservation & Anonymization**: IP/PII pseudonymization, K-anonymity scoring, and Differential Privacy Laplace mechanism.
3. **Machine Learning Security & Adversarial Robustness**: Evasion attack simulation, perturbation budgets ($\epsilon$), and feature sensitivity ranking.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ethics.fairness import FairnessAuditor
from src.ethics.privacy import PrivacyEngine
from src.ethics.security import AdversarialRobustnessAuditor
from src.pipeline import CyberRiskPipeline

pipeline = CyberRiskPipeline()
pipeline.run_data_pipeline(use_synthetic=True)
pipeline.run_training()
pipeline.run_evaluation()

print("Ethics & Security auditing suite initialized!")

### 1. Algorithmic Fairness & Demographic Parity Audit

In [ ]:
auditor = FairnessAuditor()
model = list(pipeline.trained_models.values())[0]
y_pred = model.predict(pipeline.X_test)

# Audit across user geographic cohorts
location_groups = pipeline.data["location"].iloc[:len(pipeline.y_test)].values
fairness_res = auditor.audit(pipeline.y_test, y_pred, location_groups)

print(f"Overall Fairness Status: {fairness_res['status']}")
print("Group-Level Telemetry Performance:")
display(pd.DataFrame.from_dict(fairness_res['group_metrics'], orient='index'))

### 2. Privacy Preservation & Differential Privacy

In [ ]:
privacy = PrivacyEngine()

# K-Anonymity evaluation
k_anon = privacy.audit_k_anonymity(pipeline.data, quasi_identifiers=["hour_of_day", "location", "source_type"], target_k=5)
print(f"K-Anonymity Achieved: k={k_anon['k_achieved']} (Status: {k_anon['status']})")

# Differentially Private Risk Metric Release
risk_scores = pipeline.run_risk_fusion()
dp_report = privacy.differentially_private_risk_report(risk_scores, epsilon=1.0)
print(f"True Mean Risk: {dp_report['true_mean_risk']} | Differentially Private Mean: {dp_report['differentially_private_mean_risk']}")

### 3. Adversarial Evasion Robustness Testing

In [ ]:
robustness_auditor = AdversarialRobustnessAuditor()
sec_res = robustness_auditor.evaluate_evasion_robustness(
    model=model,
    X_test=pipeline.X_test,
    y_test=pipeline.y_test,
    feature_names=pipeline.feature_names
)

print(f"Model Adversarial Robustness Score: {sec_res['robustness_score']}% ({sec_res['status']})")
pert_df = pd.DataFrame.from_dict(sec_res['perturbation_analysis'], orient='index')
display(pert_df)

plt.figure(figsize=(7, 4))
plt.plot(pert_df['epsilon'], pert_df['evasion_success_rate'], marker='s', color='crimson', label='Evasion Rate')
plt.title('Adversarial Evasion Attack Resilience Curve')
plt.xlabel('Perturbation Budget (Epsilon)')
plt.ylabel('Evasion Success Rate')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()